# Prithvi-100M — Threshold Optimization

**Environment**: Colab A100 (GPU required)

Runs inference on the validation set and sweeps thresholds 0.05→0.95 to find the operating point that maximizes F1 and IoU.
Saves `val_probs.npy` and `val_labels.npy` to Drive for future local use.

---

## Run Guide

| Cell | Description |
|---|---|
| 1 | Install dependencies |
| 2 | Drive mount & environment |
| 3 | Imports |
| 4 | Paths & constants |
| 5 | Architecture |
| 6 | Load model from Drive |
| 7 | Recreate validation split |
| 8 | Run inference & save probabilities |
| 9 | Threshold sweep & plot |

**Run all cells in order (1 → 9).**

In [ ]:
# [1] Install dependencies
import subprocess, sys, os

needs_install = False
try:
    import numpy as np
    assert tuple(int(x) for x in np.__version__.split('.')[:2]) >= (2, 0)
    from terratorch.registry import BACKBONE_REGISTRY
    print(f'OK — numpy={np.__version__}, terratorch ready.')
except Exception as e:
    needs_install = True
    print(f'Installing ({e})...')

if needs_install:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'numpy==2.0.2', '--force-reinstall'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'terratorch', 'einops', 'timm',
                    'segmentation-models-pytorch'], check=True)
    print('Restarting kernel...')
    os.kill(os.getpid(), 9)


In [ ]:
# [2] Drive mount & environment
try:
    import google.colab
    IN_COLAB = True
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    IN_COLAB = False

print(f'IN_COLAB = {IN_COLAB}')


In [ ]:
# [3] Imports
import os
import random
import subprocess
import warnings
from pathlib import Path
from tqdm import tqdm

import numpy as np
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'numpy  : {np.__version__}')
print(f'torch  : {torch.__version__}')
print(f'device : {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU    : {torch.cuda.get_device_name(0)}')


In [ ]:
# [4] Paths & constants
if IN_COLAB:
    BASE = Path('/content/drive/MyDrive/GeoAI/wildfire-spread')
else:
    BASE = Path('G:/Mon Drive/GeoAI/wildfire-spread')

CKPT    = BASE / 'models/best_prithvi_burn_scar_wildfire.pth'
RESULTS = BASE / 'results'
RESULTS.mkdir(parents=True, exist_ok=True)

if IN_COLAB:
    LOCAL = Path('/content/patches')
    LOCAL.mkdir(exist_ok=True)
    if not (LOCAL / 'images').exists():
        print('Copying patches to /content/...')
        subprocess.run(['cp', '-r', str(BASE / 'data/patches/images'),     str(LOCAL)], check=True)
        subprocess.run(['cp', '-r', str(BASE / 'data/patches/masks_dnbr'), str(LOCAL)], check=True)
        print('Done.')
    else:
        print('Patches already in /content/')
    IMG_DIR  = LOCAL / 'images'
    MASK_DIR = LOCAL / 'masks_dnbr'
else:
    IMG_DIR  = BASE / 'data/patches/images'
    MASK_DIR = BASE / 'data/patches/masks_dnbr'

BAND_IDX     = [0, 1, 2, 4, 5, 6]
PRITHVI_MEAN = np.array([0.033349, 0.057012, 0.058897, 0.232325, 0.197285, 0.119449], dtype=np.float32)
PRITHVI_STD  = np.array([0.022691, 0.026808, 0.040041, 0.077917, 0.087087, 0.072420], dtype=np.float32)
IMG_SIZE = 224
T        = (256 - IMG_SIZE) // 2
VAL_FRAC = 0.2

img_paths  = sorted(IMG_DIR.glob('*.npy'))
mask_paths = sorted(MASK_DIR.glob('*.npy'))
print(f'Patches : {len(img_paths)}')
assert CKPT.exists(), f'Checkpoint not found: {CKPT}'
print(f'Checkpoint: OK ({CKPT.stat().st_size/1e6:.0f} MB)')


In [ ]:
# [5] Architecture (identical to training notebook)
class PrithviNeck(nn.Module):
    def __init__(self, n_side=14):
        super().__init__()
        self.n_side = n_side
    def forward(self, layers_out):
        x = layers_out[-1][:, 1:, :]
        B, L, D = x.shape
        return x.permute(0, 2, 1).reshape(B, D, self.n_side, self.n_side)

class FPNDecoder(nn.Module):
    def __init__(self, in_ch):
        super().__init__()
        self.up = nn.Sequential(
            nn.ConvTranspose2d(in_ch, 256, 2, stride=2), nn.BatchNorm2d(256), nn.GELU(),
            nn.ConvTranspose2d(256,   128, 2, stride=2), nn.BatchNorm2d(128), nn.GELU(),
            nn.ConvTranspose2d(128,    64, 2, stride=2), nn.BatchNorm2d(64),  nn.GELU(),
            nn.ConvTranspose2d(64,     32, 2, stride=2), nn.BatchNorm2d(32),  nn.GELU(),
            nn.Conv2d(32, 1, 1),
        )
    def forward(self, x): return self.up(x)

class PrithviSegModel(nn.Module):
    def __init__(self, backbone, embed_dim=768, n_side=14):
        super().__init__()
        self.backbone = backbone
        self.neck     = PrithviNeck(n_side)
        self.decoder  = FPNDecoder(embed_dim)
    def forward(self, x):
        feats = self.backbone(x.unsqueeze(2))
        return self.decoder(self.neck(feats))

print('Architecture defined.')


In [ ]:
# [6] Load Prithvi backbone & model weights from Drive
from terratorch.registry import BACKBONE_REGISTRY

print('Loading prithvi_eo_v1_100...')
backbone = BACKBONE_REGISTRY.build('prithvi_eo_v1_100').to(DEVICE)
for p in backbone.parameters():
    p.requires_grad = False

model = PrithviSegModel(backbone).to(DEVICE)
model.load_state_dict(torch.load(CKPT, map_location=DEVICE))
model.eval()
print(f'Model loaded: {CKPT.name}')
print(f'Parameters  : {sum(p.numel() for p in model.parameters())/1e6:.1f}M total')


In [ ]:
# [7] Recreate validation split (same seed as training)
print('Scanning fire flags...')
fire_flags = np.array([
    np.load(p, mmap_mode='r').max() > 0 for p in tqdm(mask_paths)
], dtype=np.int32)

indices = np.arange(len(img_paths))
train_idx, val_idx = train_test_split(
    indices, test_size=VAL_FRAC, stratify=fire_flags, random_state=SEED
)

val_imgs  = [img_paths[i]  for i in val_idx]
val_masks = [mask_paths[i] for i in val_idx]

print(f'Val patches : {len(val_idx)} ({fire_flags[val_idx].sum()} with fire)')


In [ ]:
# [8] Run inference on validation set & save probabilities to Drive
print('Running inference...')
all_probs, all_targets = [], []

with torch.no_grad():
    for ip, mp in tqdm(zip(val_imgs, val_masks), total=len(val_imgs)):
        raw  = np.load(ip,  mmap_mode='r').astype(np.float32)
        mask = np.load(mp,  mmap_mode='r')

        img_norm  = raw[BAND_IDX] / 10000.0
        img_norm  = (img_norm - PRITHVI_MEAN[:, None, None]) / PRITHVI_STD[:, None, None]
        img_crop  = img_norm[:, T:T+IMG_SIZE, T:T+IMG_SIZE]
        mask_crop = mask[T:T+IMG_SIZE, T:T+IMG_SIZE].copy()

        tensor = torch.from_numpy(img_crop).unsqueeze(0).to(DEVICE)
        prob   = model(tensor).sigmoid().squeeze().cpu().numpy()

        all_probs.append(prob.flatten())
        all_targets.append(mask_crop.flatten())

all_probs   = np.concatenate(all_probs).astype(np.float32)
all_targets = np.concatenate(all_targets).astype(np.uint8)

tp = int(((all_probs > 0.5).astype(int) & (all_targets == 1)).sum())
fp = int(((all_probs > 0.5).astype(int) & (all_targets == 0)).sum())
fn = int(((all_probs <= 0.5).astype(int) & (all_targets == 1)).sum())
iou_base  = tp / (tp + fp + fn + 1e-6)
prec_base = tp / (tp + fp + 1e-6)
rec_base  = tp / (tp + fn + 1e-6)
f1_base   = 2 * prec_base * rec_base / (prec_base + rec_base + 1e-6)
print(f'Baseline t=0.50 : IoU={iou_base:.4f}  F1={f1_base:.4f}  Prec={prec_base:.4f}  Rec={rec_base:.4f}')

np.save(RESULTS / 'val_probs.npy',  all_probs)
np.save(RESULTS / 'val_labels.npy', all_targets)
print(f'Saved to {RESULTS}')


In [ ]:
# [9] Threshold sweep & plot
thresholds = np.arange(0.05, 0.96, 0.025)
results    = []

for t in thresholds:
    preds_t = (all_probs > t).astype(np.int32)
    tp = int(((preds_t==1)&(all_targets==1)).sum())
    fp = int(((preds_t==1)&(all_targets==0)).sum())
    fn = int(((preds_t==0)&(all_targets==1)).sum())
    iou_t  = tp / (tp + fp + fn + 1e-6)
    prec_t = tp / (tp + fp + 1e-6)
    rec_t  = tp / (tp + fn + 1e-6)
    f1_t   = 2 * prec_t * rec_t / (prec_t + rec_t + 1e-6)
    results.append((t, iou_t, f1_t, prec_t, rec_t))

results      = np.array(results)
best_f1_idx  = results[:, 2].argmax()
best_iou_idx = results[:, 1].argmax()

print('=' * 65)
print(f'  Baseline  t=0.500 : IoU={iou_base:.4f}  F1={f1_base:.4f}  Prec={prec_base:.4f}  Rec={rec_base:.4f}')
print(f'  Best F1   t={results[best_f1_idx,0]:.3f}  : IoU={results[best_f1_idx,1]:.4f}  F1={results[best_f1_idx,2]:.4f}  Prec={results[best_f1_idx,3]:.4f}  Rec={results[best_f1_idx,4]:.4f}')
print(f'  Best IoU  t={results[best_iou_idx,0]:.3f}  : IoU={results[best_iou_idx,1]:.4f}  F1={results[best_iou_idx,2]:.4f}  Prec={results[best_iou_idx,3]:.4f}  Rec={results[best_iou_idx,4]:.4f}')
print('=' * 65)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(results[:,0], results[:,1], label='IoU',       color='steelblue',  lw=2)
axes[0].plot(results[:,0], results[:,2], label='F1',        color='green',      lw=2)
axes[0].plot(results[:,0], results[:,3], label='Precision', color='darkorange', lw=2, ls='--')
axes[0].plot(results[:,0], results[:,4], label='Recall',    color='crimson',    lw=2, ls='--')
axes[0].axvline(0.5,                        color='gray',  ls=':',  lw=1.5, label='Baseline (t=0.50)')
axes[0].axvline(results[best_f1_idx, 0],    color='green', ls='--', lw=1.5, label=f'Best F1 (t={results[best_f1_idx,0]:.2f})')
axes[0].set(xlabel='Threshold', ylabel='Score', title='Metrics vs Threshold')
axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)

axes[1].plot(results[:,4], results[:,3], color='steelblue', lw=2)
axes[1].scatter([rec_base],               [prec_base],               color='gray',  s=80, zorder=5, label='t=0.50 (baseline)')
axes[1].scatter([results[best_f1_idx,4]], [results[best_f1_idx,3]], color='green', s=80, zorder=5, label=f't={results[best_f1_idx,0]:.2f} (best F1)')
axes[1].set(xlabel='Recall', ylabel='Precision', title='Precision-Recall Curve')
axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)

plt.suptitle('Threshold Optimization — Prithvi-100M + FPN | Corrientes val set', fontsize=11)
plt.tight_layout()
out = RESULTS / 'threshold_sweep.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {out}')
